# Feature Scaling with StandardScaler

Many machine learning models are sensitive to the scale of numerical features. When one feature ranges from 0 to 1 and another ranges from 0 to 100000, optimization can become biased toward larger-scale variables.

Feature scaling ensures numerical inputs contribute proportionally to learning. One of the most common techniques is standardization using `StandardScaler` from scikit-learn.

Scaling is not optional decoration. For many algorithms, it is a mathematical requirement.

## Why Scaling Is Necessary

Consider numerical features such as:
- `Age` (18-70)
- `MonthlyCharges` (0-150)
- `TotalCharges` (0-10000)

Without scaling:
- Larger-magnitude features dominate optimization.
- Gradient-based learning can converge slowly or unstably.
- Distance-based algorithms become distorted.

With scaling:
- The optimization landscape is smoother.
- Learning is more stable.
- Feature influence reflects signal, not raw magnitude.

## Which Models Require Scaling?

Scaling is especially important for:
- Logistic Regression
- Linear Regression (particularly Ridge/Lasso)
- Support Vector Machines (SVM)
- k-Nearest Neighbors (kNN)
- Neural Networks
- Principal Component Analysis (PCA)

Scaling is usually not required for:
- Decision Trees
- Random Forest
- Gradient Boosting (XGBoost, LightGBM, CatBoost)

Tree models are generally scale-invariant because they split on thresholds, not geometric distances.

## What Does StandardScaler Do?

For each numerical feature, StandardScaler applies:

\[
z = \frac{x - \mu}{\sigma}
\]

Where:
- `x` is the original value
- `mu` is the feature mean
- `sigma` is the feature standard deviation

After transformation:
- Mean is approximately `0`
- Standard deviation is approximately `1`

Interpretation example:
- If `MonthlyCharges` has mean `70` and std `30`, then value `100` becomes `(100 - 70) / 30 = 1`.
- Value `40` becomes `(40 - 70) / 30 = -1`.

This makes numerical features dimensionless and comparable.

## Correct Workflow: Avoiding Data Leakage

Scaling must follow this sequence:
1. Separate `X` and `y`
2. Split into train and test
3. Fit scaler on training data only
4. Transform training data
5. Transform test data using the fitted scaler

Incorrect workflow (leakage):

```python
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)  # WRONG: uses full dataset
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y)
```

Correct workflow:

```python
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
```

Never call `fit()` on test data.

## Scaling Only Numerical Features

Apply `StandardScaler` only to numerical columns.

Do not apply it to:
- Categorical features
- Binary flags without domain consideration
- Target variable (except specific regression workflows)

Example numerical list:

```python
NUMERICAL_FEATURES = [
    "tenure",
    "MonthlyCharges",
    "TotalCharges",
]
```

Categorical features should be encoded separately.

## Using ColumnTransformer (Recommended)

Use `ColumnTransformer` to combine selective scaling and categorical encoding in one reproducible pipeline.

Benefits:
- Clear preprocessing separation
- Reduced leakage risk
- Easier deployment
- Better reproducibility

In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Example mixed-type dataset
data = pd.DataFrame({
    "tenure": [1, 5, 12, 18, 24, 36, 48, 60],
    "MonthlyCharges": [29.9, 45.0, 60.0, 70.0, 80.0, 90.0, 99.5, 110.0],
    "TotalCharges": [29.9, 225.0, 720.0, 1260.0, 1920.0, 3240.0, 4776.0, 6600.0],
    "ContractType": ["Month-to-month", "Month-to-month", "One year", "One year", "Two year", "Two year", "Two year", "Two year"],
    "readmitted": [1, 1, 0, 0, 0, 0, 0, 0],
})

NUMERICAL_FEATURES = ["tenure", "MonthlyCharges", "TotalCharges"]
CATEGORICAL_FEATURES = ["ContractType"]

X = data.drop(columns=["readmitted"])
y = data["readmitted"]

# Correct sequence: split first
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Leakage-safe preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERICAL_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

model = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000)),
])

model.fit(X_train, y_train)

# Verify scaling on training numerical columns only
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train[NUMERICAL_FEATURES])
X_test_num_scaled = scaler.transform(X_test[NUMERICAL_FEATURES])

print("Train scaled mean:", np.round(X_train_num_scaled.mean(axis=0), 6))
print("Train scaled std:", np.round(X_train_num_scaled.std(axis=0), 6))
print("Test transformed shape:", X_test_num_scaled.shape)

# Save scaler artifact for production use
joblib.dump(scaler, "models/standard_scaler.pkl")
print("Saved scaler to models/standard_scaler.pkl")

## Verifying Scaling

After scaling training numerical features, verify:

```python
print(X_train_scaled.mean(axis=0))
print(X_train_scaled.std(axis=0))
```

Expected: mean close to `0`, standard deviation close to `1`.

## Common Mistakes

- Scaling before train-test split (leakage)
- Scaling categorical columns
- Calling `fit_transform` on both train and test sets
- Forgetting to persist the fitted scaler
- Re-fitting the scaler during inference

## When Not to Use StandardScaler

If you are using only tree models (Decision Tree, Random Forest, Gradient Boosting), scaling is often unnecessary. If your pipeline includes distance-based or linear components, scaling is still beneficial.

## StandardScaler vs MinMaxScaler

- `StandardScaler`: centers to mean `0`, scales to variance `1`; common for linear models and SVM.
- `MinMaxScaler`: maps to `[0, 1]`; useful where bounded input is required.

## Best Practices

- Split before scaling
- Scale only numerical features
- Prefer `Pipeline` + `ColumnTransformer`
- Save scaler with model artifacts
- Never fit on test data
- Document preprocessing choices

## Bonus Resources

- Scikit-learn StandardScaler Documentation: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
- Scikit-learn Pipeline and ColumnTransformer Guide: https://scikit-learn.org/stable/modules/compose.html